In [3]:
# This script defines functions to interact with the CADD (Combined Annotation Dependent Depletion) API to retrieve annotations for genomic variants.
# It provides options to fetch data either by specifying a single genomic variant or by processing a VCF file containing multiple variants.

import pandas as pd
import os
import io
import requests
from ipywidgets import widgets, Layout
from IPython.display import display, HTML, FileLink

# Function to read a VCF file and parse its contents into a DataFrame
def read_vcf(input_file):
    # Read the VCF file, excluding header lines (lines starting with '##')
    with open(input_file, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    # Parse the data into a DataFrame, specifying column data types and separator
    data_frame = pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
               'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    )
    return data_frame

# Function to fetch data from the CADD API for single genomic variants
def fetch_data_from_cadd(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        with output:
            output.clear_output()
            print(f"Failed to fetch data from {url}")

def CADD_SNV(data_frame):
    output_data = []
    for index, row in data_frame.iterrows():
        chrom = row['#CHROM']
        pos = row['POS']
        alt = row['ALT']
        ref = row['REF']
        url = f'https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7_inclAnno/{chrom}:{pos}_{ref}_{alt}'
        with output:
            output.clear_output()
            #print(f'Processing: {url}')
            print('Processing Data')
        response_data = fetch_data_from_cadd(url)
        if response_data:
            for item in response_data:
                item['Chrom'] = chrom
                item['Position'] = pos
                item['REF'] = ref
                item['ALT'] = alt
            output_data.extend(response_data)
    return output_data

# Function to process single input genomic variants
def process_single_input(button):
    chrom = chrom_input.value
    pos = pos_input.value
    ref = ref_input.value
    alt = alt_input.value
    url = f'https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7_inclAnno/{chrom}:{pos}_{ref}_{alt}'
    response_data = fetch_data_from_cadd(url)
    if response_data:
        df_cadd = pd.DataFrame(response_data)
        df_cadd['Chrom'] = chrom
        df_cadd['Position'] = pos
        df_cadd['REF'] = ref
        df_cadd['ALT'] = alt
        df_cadd = df_cadd[['Chrom', 'Position', 'REF', 'ALT'] + [col for col in df_cadd.columns if col not in ['Chrom', 'Position', 'REF', 'ALT']]]

        # Save to excel file
        excel_file_path = 'cadd_data_single_input.xlsx'
        df_cadd.to_excel(excel_file_path, index=False)
        with output:
            output.clear_output()
            print(f"Data saved to '{excel_file_path}'")
            display(FileLink(excel_file_path, result_html_prefix="Download Excel file: "))
            display(HTML(df_cadd.to_html(index=False, border=1)))
            
# Function to handle click event for VCF file input button
def handle_vcf_file_button_click(button):
    file_name = vcf_file_input.value.strip()
    if file_name:
        if not os.path.exists(file_name):
            with output:
                output.clear_output()
                print(f"The file '{file_name}' does not exist.")
            return
        data_frame = read_vcf(file_name)
        cadd_data = CADD_SNV(data_frame)
        if cadd_data:
            df_cadd = pd.DataFrame(cadd_data)
            df_cadd = df_cadd[['Chrom', 'Position', 'REF', 'ALT'] + [col for col in df_cadd.columns if col not in ['Chrom', 'Position', 'REF', 'ALT']]]

            # Save to excel file
            excel_file_path = 'vcf_cadd_data.xlsx'
            df_cadd.to_excel(excel_file_path, index=False)
            with output:
                output.clear_output()
                print(f"Data saved to '{excel_file_path}'")
                display(FileLink(excel_file_path, result_html_prefix="Download Excel file: "))
                display(HTML(df_cadd.to_html(index=False, border=1)))
                

    else:
        with output:
            output.clear_output()
            print("Please enter a valid file name.")
            
# Function to check if a file exists
def file_exists(file_name):
    return os.path.isfile(file_name)
    
# Dropdown options for selecting single input or VCF file data
options = ['Choose an option:', 'Single input', 'VCF file data']
dropdown = widgets.Dropdown(options=options, description='', layout=Layout(width='50%'))

output = widgets.Output()

# Event handler for dropdown selection
def dropdown_event(change):
    output.clear_output()
    if change['new'] == 'Single input':
        with output:
            display(widgets.VBox([chrom_input, pos_input, ref_input, alt_input, single_input_button]))
    elif change['new'] == 'VCF file data':
        with output:
            display(widgets.VBox([vcf_file_input, vcf_file_button]))

dropdown.observe(dropdown_event, names='value')

# Text input widgets for single input
chrom_input = widgets.Text(placeholder='1', description='Chromosome:', style={'description_width': 'initial'})
pos_input = widgets.Text(placeholder='8365860', description='Position:', style={'description_width': 'initial'})
ref_input = widgets.Text(placeholder='C', description='REF:', style={'description_width': 'initial'})
alt_input = widgets.Text(placeholder='T', description='ALT:', style={'description_width': 'initial'})
single_input_button = widgets.Button(description='Submit Single Input')
single_input_button.on_click(process_single_input)

# Text input widgets for VCF file data
vcf_file_input = widgets.Text(placeholder='Enter VCF file name', description='VCF File Name:', style={'description_width': 'initial'}, layout=Layout(width='auto'))
vcf_file_button = widgets.Button(description='Submit VCF File')
vcf_file_button.on_click(handle_vcf_file_button_click)

display(widgets.VBox([dropdown, output]))
